# P10A — Generator data sintesis beton per kelompok
Dataset ini hanya untuk menguji prosedur audit. DGP, variasi kelompok, seed, dan gangguan pengukuran dinyatakan eksplisit. Jalankan dari root proyek; notebook menghasilkan CSV dan berkas Orange .tab dengan target kuat tekan serta ID kelompok sebagai metadata. Data bukan hasil uji laboratorium dan tidak boleh dipakai untuk rancangan campuran.

In [ ]:
# Folder kerja: jalankan dari folder mana pun. Sel ini pindah ke akar repositori agar jalur data relatif (praktikum/...) benar.
import os
from pathlib import Path
_akar = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "praktikum").is_dir() and (p / "notebooks").is_dir()), None)
if _akar is not None:
    os.chdir(_akar)
else:
    print("Akar repositori tidak ditemukan. Buka notebook dari dalam folder repositori hasil clone.")

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED = 1010
rng = np.random.default_rng(SEED)
n_batches = 9
n_per_batch = 110
rows = []
batch_effects = rng.normal(0, 2.6, n_batches)
for b in range(n_batches):
    n = n_per_batch
    binder = rng.uniform(280, 500, n)              # kg/m3
    water = rng.uniform(145, 220, n)               # kg/m3
    age = rng.choice([7, 14, 28, 56, 90], n, p=[.12,.12,.42,.20,.14])
    fine_agg = rng.uniform(600, 900, n)            # kg/m3
    coarse_agg = rng.uniform(800, 1150, n)         # kg/m3
    curing_temp = rng.normal(23 + 0.4*b, 2.0, n)   # deg C
    wb = water / binder
    strength = (8 + 0.082*binder - 22*wb + 4.2*np.log(age)
                + 0.010*fine_agg + 0.004*coarse_agg
                - 0.12*np.abs(curing_temp-23) + batch_effects[b]
                + rng.normal(0, 3.0, n))
    for i in range(n):
        rows.append((f'B{b+1:02d}-{i+1:03d}', f'B{b+1:02d}', binder[i], water[i],
                     age[i], fine_agg[i], coarse_agg[i], curing_temp[i], strength[i]))

df = pd.DataFrame(rows, columns=['specimen_id','batch_id','binder_kg_m3','water_kg_m3',
                                 'age_day','fine_aggregate_kg_m3','coarse_aggregate_kg_m3',
                                 'curing_temp_C','strength_MPa'])
outdir = Path('praktikum/bab10')
outdir.mkdir(parents=True, exist_ok=True)
df.to_csv(outdir/'concrete_strength_synthetic.csv', index=False)
# Orange .tab: outcome is the class variable; IDs and batch labels are metadata.
import csv
tab_path = outdir/'concrete_strength_orange.tab'
roles = ['meta', 'meta', '', '', '', '', '', '', 'class']
types = ['string', 'string'] + ['continuous']*7
with tab_path.open('w', encoding='utf-8', newline='') as f:
    writer = csv.writer(f, delimiter='\t')
    writer.writerow(df.columns)
    writer.writerow(types)
    writer.writerow(roles)
    writer.writerows(df.itertuples(index=False, name=None))
print('rows:', len(df), '| batches:', df.batch_id.nunique(), '| seed:', SEED)
print('strength range:', round(df.strength_MPa.min(),2), 'to', round(df.strength_MPa.max(),2), 'MPa')
print(df.head(3).round(2).to_string(index=False))

rows: 990 | batches: 9 | seed: 1010
strength range: 28.27 to 78.78 MPa
specimen_id batch_id  binder_kg_m3  water_kg_m3  age_day  fine_aggregate_kg_m3  coarse_aggregate_kg_m3  curing_temp_C  strength_MPa
    B01-001      B01        422.32       192.29       14                763.34                 1095.86          20.80         48.00
    B01-002      B01        337.45       147.25       28                797.68                 1133.52          23.78         47.29
    B01-003      B01        472.18       151.36        7                677.71                  880.80          23.32         55.41
